In [1]:
# =========================================
# Import packages and set up Neo4j
# =========================================

from dotenv import load_dotenv
import os
import textwrap

# Langchain (modern)
from langchain_community.graphs import Neo4jGraph
from langchain_community.vectorstores import Neo4jVector
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

# LCEL
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Warning control
import warnings
warnings.filterwarnings("ignore")

# Load env
load_dotenv('.env', override=True)
NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE') or 'neo4j'

# Constants
VECTOR_INDEX_NAME = 'form_10k_chunks'
VECTOR_NODE_LABEL = 'Chunk'
VECTOR_SOURCE_PROPERTY = 'text'
VECTOR_EMBEDDING_PROPERTY = 'textEmbedding'

# Models
embeddings = OpenAIEmbeddings()
llm = ChatOpenAI(temperature=0)

# Neo4j
kg = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE
)

In [2]:
# =========================================
# Read the collection of Form 13s
# =========================================
import csv

all_form13s = []
with open('./data/form13.csv', mode='r') as csv_file:
    csv_reader = csv.DictReader(csv_file)
    for row in csv_reader:
        all_form13s.append(row)

print(len(all_form13s))
print(all_form13s[:5])

561
[{'source': 'https://sec.gov/Archives/edgar/data/1000275/0001140361-23-039575.txt', 'managerCik': '1000275', 'managerAddress': 'ROYAL BANK PLAZA, 200 BAY STREET, TORONTO, A6, M5J2J5', 'managerName': 'Royal Bank of Canada', 'reportCalendarOrQuarter': '2023-06-30', 'cusip6': '64110D', 'cusip': '64110D104', 'companyName': 'NETAPP INC', 'value': '64395000000.0', 'shares': '842850'}, {'source': 'https://sec.gov/Archives/edgar/data/1002784/0001387131-23-009542.txt', 'managerCik': '1002784', 'managerAddress': '1875 Lawrence Street, Suite 300, Denver, CO, 80202-1805', 'managerName': 'SHELTON CAPITAL MANAGEMENT', 'reportCalendarOrQuarter': '2023-06-30', 'cusip6': '64110D', 'cusip': '64110D104', 'companyName': 'NETAPP INC', 'value': '2989085000.0', 'shares': '39124'}, {'source': 'https://sec.gov/Archives/edgar/data/1007280/0001007280-23-000008.txt', 'managerCik': '1007280', 'managerAddress': '277 E TOWN ST, COLUMBUS, OH, 43215', 'managerName': 'PUBLIC EMPLOYEES RETIREMENT SYSTEM OF OHIO', 'r

In [4]:
# =========================================
# Create company nodes in the graph
# =========================================

# Batch creation (optimized)
kg.query("""
UNWIND $rows AS row
MERGE (com:Company {cusip6: row.cusip6})
ON CREATE SET 
    com.companyName = row.companyName,
    com.cusip = row.cusip
""", params={"rows": all_form13s})

print(kg.query("MATCH (c:Company) RETURN count(c)"))


[{'count(c)': 1}]


In [5]:
# =========================================
# Link Company → Form
# =========================================
kg.query("""
MATCH (com:Company), (form:Form)
WHERE com.cusip6 = form.cusip6
SET com.names = form.names
MERGE (com)-[:FILED]->(form)
""")

[]

In [6]:
# =========================================
# Create manager nodes
# =========================================

kg.query("""
CREATE CONSTRAINT unique_manager IF NOT EXISTS
FOR (n:Manager) REQUIRE n.managerCik IS UNIQUE
""")

kg.query("""
UNWIND $rows AS row
MERGE (mgr:Manager {managerCik: row.managerCik})
ON CREATE SET 
    mgr.managerName = row.managerName,
    mgr.managerAddress = row.managerAddress
""", params={"rows": all_form13s})

print(kg.query("MATCH (m:Manager) RETURN count(m)"))

[{'count(m)': 561}]


In [7]:
# =========================================
# Create relationships between managers and companies
# =========================================

kg.query("""
UNWIND $rows AS row
MATCH (mgr:Manager {managerCik: row.managerCik}),
      (com:Company {cusip6: row.cusip6})
MERGE (mgr)-[owns:OWNS_STOCK_IN {
    reportCalendarOrQuarter: row.reportCalendarOrQuarter
}]->(com)
ON CREATE SET 
    owns.value = toFloat(row.value),
    owns.shares = toInteger(row.shares)
""", params={"rows": all_form13s})

print(kg.query("""
MATCH (:Manager)-[o:OWNS_STOCK_IN]->(:Company)
RETURN count(o)
"""))

kg.refresh_schema()
print(textwrap.fill(kg.schema, 60))

[{'count(o)': 561}]
Node properties: Chunk {textEmbedding: LIST, f10kItem:
STRING, chunkSeqId: INTEGER, text: STRING, cik: STRING,
cusip6: STRING, names: LIST, formId: STRING, source: STRING,
chunkId: STRING} Form {cusip6: STRING, names: LIST, formId:
STRING, source: STRING} Company {cusip6: STRING, names:
LIST, companyName: STRING, cusip: STRING} Manager
{managerName: STRING, managerCik: STRING, managerAddress:
STRING} Relationship properties: SECTION {f10kItem: STRING}
OWNS_STOCK_IN {reportCalendarOrQuarter: STRING, value:
FLOAT, shares: INTEGER} The relationships:
(:Chunk)-[:NEXT]->(:Chunk) (:Chunk)-[:PART_OF]->(:Form)
(:Form)-[:SECTION]->(:Chunk) (:Company)-[:FILED]->(:Form)
(:Manager)-[:OWNS_STOCK_IN]->(:Company)


In [8]:
# =========================================
# Determine the number of investors
# =========================================

chunk_id = kg.query("""
MATCH (c:Chunk)
RETURN c.chunkId AS id LIMIT 1
""")[0]["id"]

print(chunk_id)

print(kg.query("""
MATCH (:Chunk {chunkId: $id})-[:PART_OF]->(f:Form)
RETURN f.source
""", params={"id": chunk_id}))

print(kg.query("""
MATCH (:Chunk {chunkId: $id})-[:PART_OF]->(f:Form),
      (com:Company)-[:FILED]->(f),
      (mgr:Manager)-[:OWNS_STOCK_IN]->(com)
RETURN com.companyName, count(mgr) AS investors
LIMIT 1
""", params={"id": chunk_id}))

0000950170-23-027948-item1-chunk0000
[{'f.source': 'https://www.sec.gov/Archives/edgar/data/1002047/000095017023027948/0000950170-23-027948-index.htm'}]
[{'com.companyName': 'NETAPP INC', 'investors': 561}]


In [9]:
# =========================================
# Build context for LLM
# =========================================

cypher = """
MATCH (:Chunk {chunkId: $id})-[:PART_OF]->(f:Form),
      (com:Company)-[:FILED]->(f),
      (mgr:Manager)-[o:OWNS_STOCK_IN]->(com)
RETURN mgr.managerName + " owns " + o.shares + 
       " shares of " + com.companyName + 
       " worth $" + apoc.number.format(toInteger(o.value)) AS text
LIMIT 10
"""

results = kg.query(cypher, params={"id": chunk_id})
print(textwrap.fill(results[0]["text"], 60))

Metropolitan Life Insurance Co/NY owns 9320 shares of NETAPP
INC worth $712,048,000


In [10]:
# =========================================
# Base Vector Retrieval
# =========================================

vector_store = Neo4jVector.from_existing_graph(
    embedding=embeddings,
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    index_name=VECTOR_INDEX_NAME,
    node_label=VECTOR_NODE_LABEL,
    text_node_properties=[VECTOR_SOURCE_PROPERTY],
    embedding_node_property=VECTOR_EMBEDDING_PROPERTY,
)

retriever = vector_store.as_retriever()

In [11]:
# =========================================
# Prompt + LCEL Chain
# =========================================

prompt = ChatPromptTemplate.from_template("""
Answer the question using the context below.

Context:
{context}

Question:
{question}
""")

plain_rag = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


In [12]:
# =========================================
# Graph-enhanced retrieval
# =========================================

investment_query = """
MATCH (node)-[:PART_OF]->(f:Form),
      (f)<-[:FILED]-(com:Company),
      (com)<-[o:OWNS_STOCK_IN]-(mgr:Manager)
WITH node, score, mgr, o, com
ORDER BY o.shares DESC LIMIT 10
WITH collect(
    mgr.managerName + " owns " + o.shares +
    " shares in " + com.companyName +
    " worth $" + apoc.number.format(toInteger(o.value))
) AS statements, node, score
RETURN apoc.text.join(statements, "\n") + "\n" + node.text AS text,
       score,
       {source: node.source} AS metadata
"""

vector_store_invest = Neo4jVector.from_existing_index(
    embedding=embeddings,
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    index_name=VECTOR_INDEX_NAME,
    text_node_property=VECTOR_SOURCE_PROPERTY,
    retrieval_query=investment_query,
)

retriever_invest = vector_store_invest.as_retriever()

invest_rag = (
    {"context": retriever_invest, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [13]:
# =========================================
# Compare outputs
# =========================================

q1 = "In a single sentence, tell me about Netapp."
print("\n--- Plain ---")
print(textwrap.fill(plain_rag.invoke(q1), 80))

print("\n--- Graph Enhanced ---")
print(textwrap.fill(invest_rag.invoke(q1), 80))


q2 = "In a single sentence, tell me about Netapp investors."
print("\n--- Plain ---")
print(textwrap.fill(plain_rag.invoke(q2), 80))

print("\n--- Graph Enhanced ---")
print(textwrap.fill(invest_rag.invoke(q2), 80))


--- Plain ---
NetApp is a global cloud-led, data-centric software company that helps
organizations manage applications and data across hybrid multicloud environments
with operational simplicity, flexibility, and cyber resilience.

--- Graph Enhanced ---
NetApp is a global cloud-led, data-centric software company that provides
customers with the freedom to manage applications and data across hybrid
multicloud environments, offering operational simplicity, flexibility, cyber
resilience, continuous operations, and sustainability.

--- Plain ---
Netapp investors benefit from the company's focus on enterprise storage, data
management, and cloud operations markets, as well as its strong global partner
ecosystem.

--- Graph Enhanced ---
Vanguard Group Inc., BlackRock Inc., and PRIMECAP MANAGEMENT CO/CA are among the
top investors in NetApp Inc., holding significant shares worth billions of
dollars.
